# Grupo 1 | MCDI501 - Estadística Computacional para la Toma de Decisiones

## Integrantes
- Pablo Ignacio Balbontin Constenla @pabbalbontin-maker
- Melany Esmeralda Reyes Leiva @melanyreyesy
- Ingeborg Andrea Munoz Carnot @dark452
- Mario Alejandro Lopez Pulgar @malp2203

## Descripción del problema - Fase 2 Análisis Exploratorio e Inferencial

*Proyecto*: Predicción de la Deserción y el Éxito Académico de los Estudiantes

El dataset elegido contiene información socioeconómica, académica y demográfica de estudiantes de educación superior en Portugal. El objetivo del proyecto es predecir si un estudiante se graduará, abandonará o permanecerá matriculado, lo que permite implementar intervenciones tempranas de retención.

**Dataset:** Predict Students' Dropout and Academic Success  


## Configuración del entorno

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Reproducibilidad
np.random.seed(42)

# Estilo gráfico
plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})
sns.set_style('whitegrid')

# Paleta de colores
COLORS_TARGET = {'Graduate': '#2ca02c', 'Dropout': '#d62728', 'Enrolled': '#1f77b4'}
ORDER_TARGET  = ['Graduate', 'Enrolled', 'Dropout']

print('Entorno configurado.')

Entorno configurado.


## Sección 1 - Preparación y Carga de Datos

Carga del dataset, verificación de estructura, tipos de variables, identificación, documentación de faltantes, duplicados e inconsistencias, limpieza básica y reporte inicial de calidad.

### 1.1 Carga del dataset

In [5]:
def load_data(file_path: str) -> pd.DataFrame:
    """Carga dataset raw, desde un archivo CSV

    Parámetros
    ----------
    file_path : str
        Ruta del archivo CSV utilizado como entrada.

    Retorno
    -------
    pd.DataFrame
        Datos cargados en un DataFrame.

    Excepción
    ---------
    FileNotFoundError
        Si la ruta al archivo CSV no existe. Se muestra un mensaje de error
    """
    try:
        df = pd.read_csv(file_path, sep=';')
    except FileNotFoundError:
        raise FileNotFoundError(
            f"No se encontró el archivo '{file_path}'."
            "Verificar que el archivo CSV se encuentre en data/raw."
        )
    return df

In [6]:
df = load_data('../data/raw/predict_students_dropout_and_academic_success.csv')
# Limpieza de nombres: eliminar espacios y tabs residuales
df.columns = [c.strip() for c in df.columns]
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')

print(f'\nPrimeras 3 filas:')
df.head(3)

Dataset cargado: 4,424 filas x 37 columnas

Primeras 3 filas:


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout


### 1.2 Verificación de estructura y clasificación de variables

In [7]:
# Clasificación semántica de las variables
# Nota: muchas variables codificadas como int son categorías ordinales/nominales

NUMERIC_CONTIN = [
    'Previous qualification (grade)', 'Admission grade',
    'Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)',
    'Unemployment rate', 'Inflation rate', 'GDP'
]

NUMERIC_DISCR = [
    'Age at enrollment',
    'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (approved)',
    'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (approved)',
    'Application order'
]

CATEG_NOMINAL = ['Marital status', 'Application mode', 'Course', 'Nacionality',
                 "Mother's qualification", "Father's qualification",
                 "Mother's occupation", "Father's occupation", 'Previous qualification']

CATEG_BINARY  = ['Daytime/evening attendance', 'Displaced', 'Educational special needs',
                 'Debtor', 'Tuition fees up to date', 'Gender',
                 'Scholarship holder', 'International']

TARGET = 'Target'

print('*** Clasificación de variables ***')
print(f'  Numéricas continuas : {len(NUMERIC_CONTIN)}')
print(f'  Numéricas discretas : {len(NUMERIC_DISCR)}')
print(f'  Categóricas nominales: {len(CATEG_NOMINAL)}')
print(f'  Categóricas binarias : {len(CATEG_BINARY)}')
print(f'  Variable objetivo    : {TARGET}')
print(f'\nTotal variables analizadas: {df.shape[1]}')

*** Clasificación de variables ***
  Numéricas continuas : 7
  Numéricas discretas : 6
  Categóricas nominales: 9
  Categóricas binarias : 8
  Variable objetivo    : Target

Total variables analizadas: 37


### 1.3 Diagnóstico de calidad: valores faltantes, duplicados e inconsistencias


In [8]:
# *** Valores faltantes ***
nulls = df.isnull().sum()
null_report = nulls[nulls > 0]

print('Valores faltantes por columna:')
if null_report.empty:
    print('  ==> Sin valores faltantes. Dataset completo.')
else:
    print(null_report)

# *** Duplicados ***
n_dup = df.duplicated().sum()
print(f'\nRegistros duplicados: {n_dup}')
if n_dup == 0:
    print('  ==> Sin duplicados.')

# *** Rangos y posibles inconsistencias ***
print('\n*** Rangos de variables numéricas continuas ***')
for v in NUMERIC_CONTIN:
    print(f'  {v}: [{df[v].min():.2f}, {df[v].max():.2f}]')

# Notas fuera de rango esperado [0, 20]
grade_cols = ['Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)']
for gc in grade_cols:
    out = ((df[gc] < 0) | (df[gc] > 20)).sum()
    print(f'  Valores fuera de [0,20] en "{gc}": {out}')

# Notas de admisión [95, 190]: rango válido del sistema ENES
out_adm = ((df['Admission grade'] < 95) | (df['Admission grade'] > 190)).sum()
print(f'  Valores fuera de [95,190] en "Admission grade": {out_adm}')

Valores faltantes por columna:
  ==> Sin valores faltantes. Dataset completo.

Registros duplicados: 0
  ==> Sin duplicados.

*** Rangos de variables numéricas continuas ***
  Previous qualification (grade): [95.00, 190.00]
  Admission grade: [95.00, 190.00]
  Curricular units 1st sem (grade): [0.00, 18.88]
  Curricular units 2nd sem (grade): [0.00, 18.57]
  Unemployment rate: [7.60, 16.20]
  Inflation rate: [-0.80, 3.70]
  GDP: [-4.06, 3.51]
  Valores fuera de [0,20] en "Curricular units 1st sem (grade)": 0
  Valores fuera de [0,20] en "Curricular units 2nd sem (grade)": 0
  Valores fuera de [95,190] en "Admission grade": 0


### 1.4 Reporte de calidad consolidado


In [9]:
print('═'*55)
print('   REPORTE INICIAL DE CALIDAD DEL DATASET')
print('═'*55)
print(f'  Registros totales          : {len(df):,}')
print(f'  Variables totales          : {df.shape[1]}')
print(f'  Valores faltantes          : {df.isnull().sum().sum()} (0.0%)')
print(f'  Registros duplicados       : {df.duplicated().sum()}')
print(f'  Variables continuas        : {len(NUMERIC_CONTIN)}')
print(f'  Variables discretas        : {len(NUMERIC_DISCR)}')
print(f'  Variables categóricas      : {len(CATEG_NOMINAL) + len(CATEG_BINARY)}')
print(f'  Variable objetivo (Target) : 3 clases - Dropout, Enrolled, Graduate')
print('═'*55)
print('  ==> El dataset no requiere imputación ni eliminación de filas.')
print('  ==> Se realizó limpieza de espacios en nombres de columnas.')
print('  ==> Rangos de variables dentro de valores esperados para el dominio.')

═══════════════════════════════════════════════════════
   REPORTE INICIAL DE CALIDAD DEL DATASET
═══════════════════════════════════════════════════════
  Registros totales          : 4,424
  Variables totales          : 37
  Valores faltantes          : 0 (0.0%)
  Registros duplicados       : 0
  Variables continuas        : 7
  Variables discretas        : 6
  Variables categóricas      : 17
  Variable objetivo (Target) : 3 clases - Dropout, Enrolled, Graduate
═══════════════════════════════════════════════════════
  ==> El dataset no requiere imputación ni eliminación de filas.
  ==> Se realizó limpieza de espacios en nombres de columnas.
  ==> Rangos de variables dentro de valores esperados para el dominio.
